In [ ]:
import boto3
import botocore
import functools
from IPython.core.display import display, HTML
from iterdub import iterdub as ib
from iterpop import iterpop as ip
import itertools as it
import json
import matplotlib
import matplotlib.pyplot as plt
import math
import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
import seaborn as sns
from teeplot import teeplot as tp


In [ ]:
from dishpylib.pyanalysis import calc_loglikelihoods_by_num_sets
from dishpylib.pyanalysis import count_hands_with_k_or_more_sets
from dishpylib.pyanalysis import count_hands_without_k_or_more_sets
from dishpylib.pyanalysis import estimate_interpolation_complexity
from dishpylib.pyanalysis import calc_loglikelihoods_over_set_sizes
from dishpylib.pyhelpers import get_env_context
from dishpylib.pyhelpers import get_git_revision_hash
from dishpylib.pyhelpers import make_timestamp
from dishpylib.pyhelpers import NumpyEncoder
from dishpylib.pyhelpers import preprocess_competition_fitnesses
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2025-09-13-dosed-nopouts"


In [ ]:
import pandas as pd
from scipy import stats

def fit_control_t_distns(control_df):

    na_rows = control_df['Fitness Differential Focal'].isna()
    assert all( control_df[ na_rows ]['Population Extinct'] )
    print(na_rows.sum())
    control_df['Fitness Differential Focal'].fillna(0, inplace=True,)

    res = []
    for series in control_df['Competition Series'].unique():

        series_df = control_df[ control_df['Competition Series'] == series ]

        # legacy data was mixed inside of the variant_df
        # wt_vs_wt_df = series_df.groupby('Competition Repro').filter(
        #     lambda x: (x['genome variation'] == 'master').all()
        # ).groupby('Competition Repro').first().reset_index()

        # fit a t distribution to the control data
        # df is degrees of freedom
        df, loc, scale = stats.t.fit( series_df['Fitness Differential Focal'] )


        res.append({
            'Series' : series,
            'Fit Degrees of Freedom' : df,
            'Fit Loc' : loc,
            'Fit Scale' : scale,
        })

    return pd.DataFrame(res)


In [ ]:
import boto3
import botocore
import functools
import pandas as pd


@functools.lru_cache
def get_control_t_distns( bucket, endeavor, stint ):

    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    control_competitions, = [*bucket_handle.objects.filter(
        Prefix=f'endeavor={endeavor}/control-competitions-focalbb-phenotypeneutral/stage=3+what=collated/stint={stint}',
    )][:1]

    control_df = pd.read_csv(
        f's3://{bucket}/{control_competitions.key}',
    )

    res = fit_control_t_distns(control_df[
        control_df["Root ID"] == 0
    ].copy())
    return res


In [ ]:
import functools
from iterpop import iterpop as ip
from scipy import stats


def preprocess_competition_fitnesses(competitions_df, control_fits_df):
    # preprocess data
    @functools.lru_cache
    def h0_fit(series):
        return ip.popsingleton(
            control_fits_df[control_fits_df["Series"] == series].to_dict(
                orient="records",
            )
        )

    competitions_df["p"] = competitions_df.apply(
        lambda row: stats.t.cdf(
            row["Fitness Differential Focal"],
            h0_fit(row["genome series"])["Fit Degrees of Freedom"],
            loc=h0_fit(row["genome series"])["Fit Loc"],
            scale=h0_fit(row["genome series"])["Fit Scale"],
        ),
        axis=1,
    )
    competitions_df["Is Less Fit"] = competitions_df["p"] < 1.0 / 40
    competitions_df["Is More Fit"] = competitions_df["p"] > (1.0 -  1.0 / 40)
    competitions_df["Is Neutral"] = ~(
        competitions_df["Is Less Fit"] | competitions_df["Is More Fit"]
    )
    competitions_df["Relative Fitness"] = competitions_df.apply(
        lambda row: (
            "Significantly Advantageous"
            if row["Is More Fit"]
            else (
                "Significantly Deleterious" if row["Is Less Fit"] else "Neutral"
            )
        ),
        axis=1,
    )

    return competitions_df


# get data


In [ ]:
s3_handle = boto3.resource(
    's3',
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket('prq49')

dfs = []
for stint in range(101):
    print(stint)
    series_profiles = [
        x for x in bucket_handle.objects.filter(
            Prefix=f'endeavor=16/bioticbackground-noncritical-phenotypeneutral-nopinterpolation-competitions/stage=7+what=generated/stint={stint}/series=16005/a=coalescence_result',
        )
        if x.key.endswith(".csv")
    ]

    control_fits_df = get_control_t_distns('prq49', 16, stint)
    for series_profile in series_profiles:
        df = pd.read_csv(
            f's3://prq49/{series_profile.key}',
        )
        df = df[df["Competition Series"] == 16005]
        df = df[df["genome morph"] != "wildtype"].copy()
        df = df[df["genome a"] == "genome"].copy()

        print(stint, len(df))
        df["Stint"] = stint
        dfdigest = '{:x}'.format( hash_pandas_object( df ).sum() )
        print(dfdigest)
        df = preprocess_competition_fitnesses(df, control_fits_df)
        dfs.append(df)


In [ ]:
df = pd.concat(dfs)
# df = pd.read_csv("2025-09-13-dosed-nopouts.csv")
# df.to_csv("2025-09-13-dosed-nopouts.csv")


In [ ]:
dfx = df[df["genome variation"] != "master"]


In [ ]:
dfx["knockout dose"] = dfx["genome nop_interpolation_num_nopped"].astype(int)


In [ ]:
get_control_t_distns('prq49', 16, 90)


In [ ]:
sns.countplot(
    hue="knockout dose",
    x="Stint",
    data=dfx.astype(
        {
            "Is More Fit": int,
            "knockout dose": int,
            "Stint": int,
        },
    ),
)
plt.show()


In [ ]:
dfx["Log dose"] = np.log1p(dfx["knockout dose"])


In [ ]:
for stint in range(5):
    sns.regplot(
        y="Is Less Fit",
        x="knockout dose",
        data=dfx[
            dfx["Stint"] // 20 == stint
        ].astype(
            {
                "Is More Fit": int,
                "knockout dose": int,
                "Stint": int,
            },
        ),
        logistic=True,
    )
plt.show()


In [ ]:
dfx["epoch_"] = dfx["Stint"] // 33
dfx["dose class"] = dfx["knockout dose"] // 10


In [ ]:
sns.lineplot(
    data=dfx[
        (dfx["Stint"] < 99)
        & (dfx["knockout dose"] != 30)
    ],
    y="Is Less Fit",
    x="dose class",
    hue="epoch_",
)
plt.show()


In [ ]:
import numpy as np
from sklearn.isotonic import IsotonicRegression

def estimate_ed50(results):
    """
    Estimate ED50 (site count with ~50% success probability) from
    31-length list of booleans (True=success, False=failure).

    results: list of length 31, each entry is True/False (success/failure).
             Index corresponds to site count (0..30).

    Returns: (ed50_discrete, ed50_interp)
      - ed50_discrete = smallest site count where fitted prob >= 0.5 (or np.inf if never reaches 0.5)
      - ed50_interp   = linear interpolation between steps, for smoother estimate
    """
    print(len(results))
    if len(results) != 31:
        raise ValueError("results must be a list of length 31 (site counts 0..30).")

    x = np.arange(31)  # site counts 0..30
    y = np.array(results, dtype=int)

    # observed success rate is just 0 or 1 at each site count
    r = y
    w = np.ones_like(r)  # each point weight =1 since one experiment per site

    iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds='clip')
    p_hat = iso.fit_transform(x, r, sample_weight=w)

    # find discrete ED50
    cross = np.where(p_hat >= 0.5)[0]
    # ed50_discrete = x[cross[0]] if cross.size else np.inf

    # interpolated ED50
    if cross.size and cross[0] > 0:
        i = cross[0]
        x0, x1 = x[i-1], x[i]
        p0, p1 = p_hat[i-1], p_hat[i]
        if p1 > p0:
            ed50_interp = x0 + (0.5 - p0) * (x1 - x0) / (p1 - p0)
        else:
            ed50_interp = x1
    elif cross.size:
        ed50_interp = x[cross[0]]
    else:
        ed50_interp = np.inf

    return ed50_interp


In [ ]:
dosages = [
    estimate_ed50(
        dfx.loc[
            dfx["Stint"] == stint,
            ["knockout dose", "Is Less Fit"],
        ].sort_values("knockout dose")["Is Less Fit"].tolist()
    )
    for stint in range(2, 101)
    if stint in dfx["Stint"].values

]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Lineplot with markers
sns.lineplot(
    x=sorted(dfx["Stint"].unique())[2:],
    y=dosages,
    marker="o"
)

# Add small text labels for each point
for x, y in zip(sorted(dfx["Stint"].unique())[2:], dosages):
    plt.text(x, y, str(x), fontsize=8, ha='right', va='bottom')

plt.show()


In [ ]:
# dfbc = pd.read_csv("biotic-complexity.csv")
dfbc = pd.read_csv("https://osf.io/xn53d/download")


In [ ]:
sns.lineplot(
    x=sorted(dfx["Stint"].unique())[2:],
    y=np.array(dosages) * 4 + 100,
    marker="o"
)
sns.lineplot(
    data=dfbc,
    x="Stint",
    y="Is Neutral",
    marker="o",
    color="orange",
)
plt.gca().set_ylim(0, 300)


In [ ]:
dfbcx = dfbc[dfbc["Stint"] >= 2].copy().set_index("Stint")

ys = dosages / (1 - (200 - dfbcx.loc[
    sorted(dfx["Stint"].unique())[2:],
    "Is Neutral"
]) / 200)

sns.lineplot(
    x=sorted(dfx["Stint"].unique())[2:],
    y=ys,
    marker="o"
)

plt.gca().set_ylim(0, 50)
plt.gca().invert_yaxis()
# Add small text labels for each point
for x, y in zip(sorted(dfx["Stint"].unique())[2:], ys):
    plt.text(x, y, str(x), fontsize=8, ha='right', va='bottom')


In [ ]:
ys = dosages[33:] / (1 - (200 - dfbcx.loc[
    sorted(dfx["Stint"].unique())[35:],
    "Is Neutral"
]) / 200)
sns.regplot(
    x=[*range(len(ys))],
    y=ys,
)


In [ ]:
from scipy import stats

# Remove NaNs if present
ys_clean = ys.dropna()

# Parametric: Pearson correlation with Stint
pearson_corr, pearson_p = stats.pearsonr(ys_clean.index, ys_clean.values)

# Nonparametric: Spearman correlation with Stint
spearman_corr, spearman_p = stats.spearmanr(ys_clean.index, ys_clean.values)

print(f"Pearson r: {pearson_corr:.3f}, p={pearson_p:.3g}")
print(f"Spearman rho: {spearman_corr:.3f}, p={spearman_p:.3g}")


In [ ]:
dfbcx = dfbc[dfbc["Stint"] >= 2].copy().set_index("Stint")

ys = dosages / (1 - (200 - dfbcx.loc[
    sorted(dfx["Stint"].unique())[2:],
    "Is Neutral"
]) / 200)

sns.lineplot(
    x=sorted(dfx["Stint"].unique())[2:],
    y=ys,
    marker="o"
)

sns.lineplot(
    x=sorted(dfx["Stint"].unique())[2:],
    y=dfbcx.loc[
        sorted(dfx["Stint"].unique())[2:],
        "Is Less Fit"
    ],
)
plt.gca().set_ylim(0, 100)


In [ ]:
sns.regplot(
    x=sorted(dfx["Stint"].unique())[2:],
    y=ys,
)


In [ ]:
sns.scatterplot(
    data=dfx.loc[
        dfx["knockout dose"] == 1,
    ],
    x="Stint",
    y="Relative Fitness",
)
